# Linear-equivalent disc / annulus with cone linearization

Python version of `linConeMain.m` + `analyzeLinearDiscCone.m` + `populationLinConeDisc.m`
(linCone repo). Same structure as the other single-cell notebooks — discover, group, analyze,
store, population figure — and sharing their light-level helpers.

**The experiment.** Every natural-image patch is shown three ways, interleaved:

| `stimulusTag` | what it is |
|---|---|
| `image` | the image patch itself |
| `intensity` | uniform disc at the linear-equivalent intensity (patch averaged over the RF) |
| `linConeIntensity` / `lin cone intensity` | uniform disc at the **cone-linearized** equivalent intensity — averaged after a Weber cone nonlinearity `I / (I + WeberConstant)` |

Comparing image against each disc asks how much of the cell's preference for the real image
survives when the averaging happens in cone-response space instead of intensity space. The
measure is the nonlinearity index per patch, exactly as `computeNLI` defines it:

$$\mathrm{NLI} = \frac{\mathrm{image} - \mathrm{disc}}{|\mathrm{image}| + |\mathrm{disc}|}$$

set to zero when neither response clears a recording-mode threshold (3 spikes extracellular,
10 exc, 5 inh).

**Three protocols feed this, and one needs filtering.** `LinearEquivalentDiscConeLin` (disc over
the centre) and `LinearEquivalentAnnulus` (disc over the surround, usually ON parasols) always
carry a `linearizeCones` parameter. `LinearEquivalentDisc` does not: that name was reused for an
older experiment, so only blocks that *have* a `linearizeCones` parameter belong here.
`find_blocks` applies that filter per block and reports what it dropped.

In [ ]:
import retinanalysis as ra
import numpy as np
import pandas as pd

from retinanalysis.SCutils import explore as sc
from retinanalysis.SCutils.protocols import linear_equivalent_disc as led

## 1. Find the blocks, applying the `linearizeCones` filter

The three protocols are pooled into one table. Watch the dropped count — that is the older
same-named protocol being excluded.

In [ ]:
df_blocks = led.find_blocks()

In [ ]:
print('disc site x cell type (blocks)')
display(pd.crosstab(df_blocks['cell_type_short'], df_blocks['site']))

print('\nrecording mode x how the cell was actually recorded')
display(pd.crosstab(df_blocks['onlineAnalysis'], df_blocks['recording_technique']))

`onlineAnalysis` was left at `none` for a good fraction of blocks. Those are resolved from how
the cell was actually recorded — cell-attached becomes spike counting, whole-cell is treated as
current with its polarity read off the data, which is what the MATLAB does.

## 2. Group into recordings

One row per experiment × cell × recording mode × disc site × light level. Blocks in a group are
pooled, so a cell recorded across several blocks at one light level yields one set of patches.

In [ ]:
groups = led.group_blocks(df_blocks)

## 3. Analyze one recording

Per epoch the onset and offset responses are measured, then averaged within
(image, patch, stimulus category). A patch is kept only if it has an image trial and at least
one disc trial. Left: each patch's image response against its two discs, with the unity line —
points above it are patches where the real image drove the cell more than the equivalent
uniform disc. Right: the NLI values those produce.

In [ ]:
row = groups.sort_values('epochs', ascending=False).iloc[0]
rec = led.analyze_group(row.exp_name, [int(b) for b in row.block_ids.split(',')],
                        online_analysis=row.onlineAnalysis)
led.plot_group(rec);

## 4. Batch analyze and save

Records go to `<OUTPUT_DIR>/linear_equivalent_disc/records.h5` with a `summary.csv` index,
upserted per (experiment, cell, mode, site, filter wheel, background) — same layout as the other
protocols, in its own directory. `skip_existing` makes re-running the notebook cheap.

In [ ]:
records = led.analyze_all(groups, save=True, plot=False, skip_existing=True)

## 5. Population: does cone linearization remove the nonlinearity?

Section 1 of `populationLinConeDisc.m`: one line per recording joining its mean standard-disc
NLI to its mean cone-linearized NLI, per recording mode, with a paired Wilcoxon signed-rank
test. If cone linearization captures what the cell is doing, the cone-linearized NLI should sit
closer to zero.

In [ ]:
summary = led.load_summary()
print(f'{len(summary)} stored recordings')
led.plot_population_nli(summary, window='onset');

In [ ]:
led.plot_population_nli(summary, window='offset');

### Pooled per-patch distributions

Section 2 of the population script — every patch from every recording, no per-cell averaging,
with a two-sample KS test.

In [ ]:
led.plot_nli_distributions(window='onset');

### Split by disc site

The annulus protocol puts the disc over the surround; the other two put it over the centre.
Those are different experiments and are worth looking at separately.

In [ ]:
for site in sorted(summary['site'].unique()):
    sub = summary[summary['site'].eq(site)]
    print(f'--- disc over {site}: {len(sub)} recordings ---')
    display(sub.groupby('online_analysis')[['nli_disc_onset', 'nli_cone_onset',
                                            'nli_disc_offset', 'nli_cone_offset']]
               .agg(['count', 'mean']).round(3))

## 6. Come back later

`load_summary()` reads the scalar index with no DataJoint or SSD access; `load_records()` pulls
the per-patch arrays.

In [ ]:
sc.scroll_table(summary[['exp_name', 'cell_label', 'cell_type', 'online_analysis',
                         'site', 'light_setting', 'n_patches',
                         'nli_disc_onset', 'nli_cone_onset']], height=320)

## Inspect one cell

Everything above runs across the whole dataset. This section goes the other way: name a single
cell and see every recording of it, split by condition.

A cell is identified by `'<experiment>/<cell label>'`, e.g. `'2026-04-03_E/Cell4'`. `list_cells` shows the
available ids together with the conditions each cell was recorded in — recording mode, site,
light setting, and whatever else varies for this protocol — so you can pick one. A bare cell
label also works when it is unambiguous.

In [ ]:
led.list_cells(groups)

In [ ]:
# Change CELL to inspect a different one. Each condition is analyzed and
# plotted in turn, and the records are returned for further poking.
CELL = '2026-04-03_E/Cell4'

cell_records = led.inspect_cell(CELL, groups)